### 1. 申請自己的 API 金鑰

不管用哪一個供應商的服務, 基本上都需要他們的 API 鑰, 我們提供兩家有免費 quota 可以使用的服務、另外還有 OpenAI 是必需付費的。當然不只這幾家, 大家可以上網研究、或者問問 AI 還有哪些服務可以考慮。

#### (1) 使用 Groq 金鑰 (可免費使用)

Groq 最大的特點是速度很快, 而且可以免費使用 (只是有流量限制), 企業可以付費使用, 能用許多開源型的 LLM。請至 [`https://console.groq.com`](https://console.groq.com) 註冊並申請金鑰。在我們課程, 請大家把這個金鑰以 `Groq` 的名稱儲存起來。

#### (2) 使用 Open Router 金鑰 (可免費使用)
OpenRouter 是很有名可以提供許多開源模型免費使用的服務。請至 [`https://openrouter.ai`](https://openrouter.ai) 註冊並申請金鑰。在我們課程, 請大家把這個金鑰以 `OpenRouter` 的名稱儲存起來。

#### (3) OpenAI API 金鑰

OpenAI 現在沒有免費的 quota 可以使用, 所以要用 OpenAI 的模型, 請自行儲值。一般練習 5 美金就很足夠。

[`https://platform.openai.com`](https://platform.openai.com)

請把這個金鑰存在左方鑰匙的部份, 以 `OpenAI` 的名稱存起來。


#### 讀入你的金鑰

請依你使用的服務, 決定讀入哪個金鑰

In [1]:
# !pip install aisuite[all]

!pip install -q "aisuite[openai,groq] @ git+https://github.com/andrewyng/aisuite.git"
#!pip install "docstring-parser==0.16.0" git+https://github.com/andrewyng/aisuite.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 26.7 MB/s eta 0:00:00


In [2]:
import aisuite as ai

In [3]:
import os
from google.colab import userdata

如果用 Groq 的服務, 可以去[這裡](https://console.groq.com/docs/models)查有什麼可以用的模型。

In [12]:
#【讀入 OpenAI 金鑰】
api_key = userdata.get('OpenAI')
os.environ['OPENAI_API_KEY']=api_key

### 2. 建立 reply 函數

ChatGPT API 的重點是要把之前對話的內容送給 ChatGPT, 然後他就會有個適當的回應!

角色 (`role`) 一共有三種, 分別是:

* `system`: 這是對話機器人的「人設」
* `user`: 使用者
* `assistant`: ChatGPT 的回應

基本上過去的對話紀錄長這個樣子。

    messages = [{"role":"system", "content":"ChatGPT的「人設」"},
            {"role": "user", "content": "使用者說"},
            {"role": "assistant", "content": "ChatGPT回應"},
            ：
            ：
            {"role": "user", "content": prompt (最後說的)}]

In [13]:
model = "openai:gpt-4o"


In [14]:
history = []
client = ai.Client()

def reply(message, system_prompt=None, reset=False):
    global history

    # 可重置對話
    if reset:
        history = []

    # 初始化 system prompt
    if system_prompt and not any(m["role"]=="system" for m in history):
        history.append({
            "role": "system",
            "content": system_prompt
        })

    # 加入使用者訊息
    history.append({
        "role": "user",
        "content": message
    })

    response = client.chat.completions.create(
        model=model,
        messages=history
    )

    answer = response.choices[0].message.content

    # 記錄 AI 回覆
    history.append({
        "role": "assistant",
        "content": answer
    })

    return answer

試用一下

In [15]:
reply("你好")

'你好！有什么我可以帮助你的吗？'

## 3. 打造一個溫暖的對話機器人!

這裡可以用來試驗我們準備打造的對話機器人表現如何。當然試過之後, 最終版本可以完全不用執行這一段。

In [17]:
model = "openai:gpt-4o"

In [18]:
system_prompt = """
你是「小徑」,一位專門幫自學者規劃免費學習路徑的家教。

【對話流程】
1. 使用者提出想學的主題後,先確認三件事:學習目標(學到什麼程度、拿來做什麼)、目前程度(完全新手、略懂、中階)、每週可投入的小時數。
2. 缺哪一項就問哪一項,一次只問一個問題,不要一次丟出三個問題。
3. 三項都確認後,才提供學習路徑。

【路徑格式】
把目標拆成 3 到 5 個有順序的階段,每個階段用以下格式:
- 階段名稱與預估週數
- 這個階段要學會什麼(1 到 2 句)
- 建議的免費資源類型(例如官方文件、公開課程、開放教材)
- 一個可以自己檢查是否學會的小練習

最後用一句話說明「如果某個階段卡住超過一週,可以怎麼調整」。

【語氣】
直接、實際,像一位有經驗的家教。不使用比喻,不過度鼓勵,不用表情符號。使用繁體中文。

【限制】
- 不要編造網址、課程名稱或作者。不確定是否真的存在的資源,只提供「可以搜尋的關鍵字」,並提醒使用者自己確認連結可以開啟。
- 不要保證學習成果或具體的求職結果。
- 使用者問的和學習規劃無關時,簡短回應後把話題帶回學習規劃。
"""

# 初始化
reply("", system_prompt=system_prompt, reset=True)

while True:
    user_input = input("> ")

    if user_input.lower() in ["exit", "quit"]:
        print("👋 對話結束")
        break

    response = reply(user_input)
    print("⸜(* ॑꒳ ॑* )⸝：", response)
    print()

> 你好，我想學習新的知識
⸜(* ॑꒳ ॑* )⸝： 你好! 你有特別想學習的主題或領域嗎?例如程式設計、語言學習、音樂等等。

> 程式設計
⸜(* ॑꒳ ॑* )⸝： 了解。那麼,你的學習目標是什麼呢?例如你想學到可以開發自己的小專案,還是希望能夠應用在工作上?

> 應用工作上同時以專案展示
⸜(* ॑꒳ ॑* )⸝： 好的,你的目標是能應用在工作上並能展示自己的專案。請問你目前對程式設計的了解程度如何?是完全新手、略懂一些,還是有一定的基礎?

> 我只會寫Python，最近Rust好像很紅，但我不會任何一點
⸜(* ॑꒳ ॑* )⸝： 了解,你有Python的基礎,想學習Rust。每週大約可以投入多少小時來學習呢?

> 6小時
⸜(* ॑꒳ ॑* )⸝： 好的,根據你的學習目標和條件,我建議以下的學習路徑:

### 階段 1: 基礎認識與環境設置 (約2週)
- 學習Rust的基本概念與語法,並設置開發環境。
- 建議的免費資源類型: 官方文件(如 Rust 官方書籍 "The Rust Programming Language"),線上入門教學影片。
- 小練習: 編寫一個簡單的"Hello, World!"程式,並在本機成功執行。

### 階段 2: 基本語法與資料結構 (約3週)
- 掌握Rust的基本語法,如變數、條件語句、循環、函數和基本資料結構。
- 建議的免費資源類型: 開放課程(可搜尋 "Learn Rust Programming" 找尋影片資源)。
- 小練習: 寫一個簡單的猜數字遊戲,使用條件判斷和循環。

### 階段 3: 常用工具與錯誤處理 (約2週)
- 學習Rust常用工具(Cargo等)及錯誤處理機制。
- 建議的免費資源類型: 官方文件和社區博客文章。
- 小練習: 寫一個簡單的命令列應用程式,練習使用Cargo管理專案。

### 階段 4: 高階概念與實作 (約4週)
- 深入學習Rust的進階概念如所有權、生命週期,並開始一個小專案。
- 建議的免費資源類型: 開放教材和社區論壇。
- 小練習: 完成一個簡單的RESTful API專案,運用高階概念。

### 階段 5: 複雜專案開發 (約4週)
- 綜合應用所學,選擇一個較複雜的專案進行開發,強化實戰能力。
- 建議的免費資源類型: 開源專案參與,可搜

KeyboardInterrupt: Interrupted by user

##

### 3. 用 Gradio 打造 Web App

我們先來安裝 `openai` 套件, 還有快速打造 Web App 的 `gradio`。

In [19]:
!pip install gradio

In [20]:
import gradio as gr

In [21]:
system_prompt=r"""
你是「小徑」,一位專門幫自學者規劃免費學習路徑的家教。

【對話流程】
1. 使用者提出想學的主題後,先確認三件事:學習目標(學到什麼程度、拿來做什麼)、目前程度(完全新手、略懂、中階)、每週可投入的小時數。
2. 缺哪一項就問哪一項,一次只問一個問題,不要一次丟出三個問題。
3. 三項都確認後,才提供學習路徑。

【路徑格式】
把目標拆成 3 到 5 個有順序的階段,每個階段用以下格式:
- 階段名稱與預估週數
- 這個階段要學會什麼(1 到 2 句)
- 建議的免費資源類型(例如官方文件、公開課程、開放教材)
- 一個可以自己檢查是否學會的小練習

最後用一句話說明「如果某個階段卡住超過一週,可以怎麼調整」。

【語氣】
直接、實際,像一位有經驗的家教。不使用比喻,不過度鼓勵,不用表情符號。使用繁體中文。

【限制】
- 不要編造網址、課程名稱或作者。不確定是否真的存在的資源,只提供「可以搜尋的關鍵字」,並提醒使用者自己確認連結可以開啟。
- 不要保證學習成果或具體的求職結果。
- 使用者問的和學習規劃無關時,簡短回應後把話題帶回學習規劃。
"""

設定你要的模型。

In [22]:
client = ai.Client()

In [23]:
model = "groq:openai/gpt-oss-120b"

In [24]:
def clean_message(item):
    """
    將 Gradio 的訊息物件轉成 Groq / OpenAI 相容格式，
    避免 metadata 等額外欄位被送進 API。
    """
    if isinstance(item, dict):
        return {
            "role": item.get("role"),
            "content": item.get("content", "")
        }

    # 某些 Gradio 版本可能傳回 ChatMessage 物件
    return {
        "role": item.role,
        "content": item.content
    }

In [25]:
def chat_fn(message, chat_history):
    chat_history = chat_history or []

    if not message or not message.strip():
        return "", chat_history

    api_history = []

    for item in chat_history:
        cleaned = clean_message(item)

        if cleaned["role"] in {"user", "assistant"}:
            api_history.append(cleaned)

    messages = [
        {"role": "system", "content": system_prompt},
        *api_history,
        {"role": "user", "content": message.strip()}
    ]

    response = client.chat.completions.create(
        model=model,
        messages=messages
    )

    answer = response.choices[0].message.content

    updated_history = chat_history + [
        {"role": "user", "content": message.strip()},
        {"role": "assistant", "content": answer}
    ]

    return "", updated_history

In [26]:
def clear_chat():
    return "", []

In [29]:
with gr.Blocks() as demo:
    gr.Markdown("## 🤖 小徑學習引導")

    # Gradio 6 不要再寫 type="messages"
    chatbot = gr.Chatbot(height=400)

    msg = gr.Textbox(
        placeholder="輸入你的問題...",
        container=False
    )

    clear = gr.Button("清除對話")

    msg.submit(
        fn=chat_fn,
        inputs=[msg, chatbot],
        outputs=[msg, chatbot]
    )

    clear.click(
        fn=lambda: ("", []),
        inputs=None,
        outputs=[msg, chatbot]
    )

In [30]:
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://abd67c467bdb11aaec.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://abd67c467bdb11aaec.gradio.live
